# Gemma 2 2B Unsloth: offline assets
Run with **Internet On**, using the same Kaggle Python/GPU image as training.
Save a completed version under `name-that-llm-gemma2b-unsloth-setting`, then attach its output to training.
This bundle contains the original 4-bit model and all resolved dependency wheels (including the GPU runtime wheels). Allow several GB of output storage. Unlike the older setup, it does not delete dependencies needed by the resolver.

In [ ]:
import sys, json, platform, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
ROOT = Path('/kaggle/working')
WHEELS = ROOT / 'wheels'
WHEELS.mkdir(exist_ok=True)
# Match Kaggle's existing GPU stack instead of resolving an unrelated Torch build.
runtime = {}
for name in ['torch', 'torchvision', 'torchaudio', 'triton']:
    try:
        runtime[name] = version(name)
    except PackageNotFoundError:
        pass
constraints = ROOT / 'runtime-constraints.txt'
constraints.write_text(''.join(f'{k}=={v.split("+")[0]}\n' for k,v in runtime.items()))
print('Runtime constraints:', runtime)
subprocess.run([sys.executable, '-m', 'pip', 'wheel', '--wheel-dir', str(WHEELS),
                '--constraint', str(constraints), 'unsloth', 'pandas', 'scikit-learn', 'pyarrow'], check=True)

In [ ]:
from packaging.utils import parse_wheel_filename
# Freeze every resolved wheel, so training uses the exact downloaded package set.
resolved = {}
for wheel in WHEELS.glob('*.whl'):
    name, ver, _, _ = parse_wheel_filename(wheel.name)
    if name in resolved and resolved[name] != str(ver):
        raise RuntimeError(f'Multiple versions of {name} in wheels. Start with a fresh Kaggle working directory.')
    resolved[name] = str(ver)
(ROOT / 'offline-requirements.txt').write_text(''.join(f'{k}=={v}\n' for k,v in sorted(resolved.items())))
manifest = {'python': list(sys.version_info[:2]), 'runtime': runtime,
            'packages': resolved, 'model_dir': 'gemma-2-2b-it-bnb-4bit'}
(ROOT / 'gemma2b-unsloth-bundle.json').write_text(json.dumps(manifest, indent=2))
print(f'Prepared {len(resolved)} package wheels.')

In [ ]:
import shutil
from huggingface_hub import snapshot_download
MODEL_DIR = ROOT / 'gemma-2-2b-it-bnb-4bit'
# Reuse the original setup output if attached; otherwise download the same repository.
candidates = [p for p in Path('/kaggle/input').rglob('gemma-2-2b-it-bnb-4bit')
              if (p/'config.json').is_file()]
if candidates:
    shutil.copytree(candidates[0], MODEL_DIR, dirs_exist_ok=True)
    print('Copied original model:', candidates[0])
else:
    snapshot_download(repo_id='unsloth/gemma-2-2b-it-bnb-4bit', local_dir=str(MODEL_DIR))
print('Ready to save notebook output:', ROOT)